In [1]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

import sys
import torch
from tqdm import tqdm
from nnsight import LanguageModel

sys.path.append("./mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder
from transformer_lens import utils

torch.set_grad_enabled(False)

In [2]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir="./jbloom_dictionaries")

    save_path = f"./jbloom_dictionaries/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [3]:
gpt = LanguageModel("openai-community/gpt2", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

Loaded GPT


In [4]:
kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

# mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)
mixtral = LanguageModel("mistralai/Mistral-7B-Instruct-v0.2", device_map="cuda:0", dispatch=True)

print("Loaded Mixtral in 4bit")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded Mixtral in 4bit


In [5]:
import simulator
import importlib 
importlib.reload(simulator)

env = simulator.Environment(mixtral, gpt, sae_list)

In [ ]:
# location = simulator.Location(
#     feature_type = "resid",
#     layer = 10,
#     index = 12307
# )

# prompts = [
#     " broadcast.ĊĊWhile the episode will mark C.K.'s debut as an",
#     "ĊĊIf implemented, this project would mark a turning point in the growing cooperation between",
#     " States.ĊĊThis year,which marks Fort RossâĢĻ bicentennial,"
# ]

In [30]:
location = simulator.Location(
    feature_type = "resid",
    layer = 10,
    index = 6536
)

prompts = [
    " (getting back 87 cents on the dollar in 2010.) In comparison, the average state gets",
    " (Dungeons and Dragons figurines off the kitchen table).ĊĊThe other day I noteds",
    ", (appears to be in much the same boat.) Yes, our political leaders are perfectly",
]

prompts = [gpt.tokenizer.bos_token + i for i in prompts]

features = []

layer = location.layer
index = location.index

from circuitsvis.tokens import colored_tokens
figs = []

for prompt in prompts:
    tokens = gpt.tokenizer.encode(prompt)
    str_tokens = [gpt.tokenizer.decode([t]) for t in tokens]

    with gpt.trace(tokens):
        activations = gpt.transformer.h[layer].input[0][0].save()

        _, feature_acts, _, _, _, _ = sae_list[layer](activations)

        acts = feature_acts[:,:,index][0].save()

    acts[0] = 0.
    acts = acts.value

    figs.append(colored_tokens(str_tokens, acts))

    f = simulator.Feature(
        prompt=prompt,
        tokens=str_tokens,
        acts=acts,
        n_acts=simulator.normalize_acts(acts),
        location=location
    )

    features.append(f)

In [40]:
features[2]

Feature(prompt='<|endoftext|>, (appears to be in much the same boat.) Yes, our political leaders are perfectly', tokens=['<|endoftext|>', ',', ' (', 'app', 'ears', ' to', ' be', ' in', ' much', ' the', ' same', ' boat', '.)', ' Yes', ',', ' our', ' political', ' leaders', ' are', ' perfectly'], acts=tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        2.9111, 2.4892, 9.9107, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000], device='cuda:0'), n_acts=tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  2.9373,  2.5116, 10.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000], device='cuda:0'), location=Location(feature_type='resid', index=6536, layer=10))

In [7]:
import sys

with open('./logs/running.log', 'w') as log_file:
    sys.stdout = log_file
    sys.stderr = log_file
    out = env(features)
    sys.stdout = sys.__stdout__ 

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Content                                                                                                  ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ User      │ You are a meticulous AI researcher conducting a high-stakes investigation on neurons in a large language │
│           │ model. Your task is to understand what features of the input text cause a specific neuron to activate.   │
│           │                                                                                                          │
│           │ You will be given a list of text samples containing tokens on which the neuron activates strongly. The   │
│           │ specific tokens which caused the neuron to activate strongly will appear between bars like | this|. If   │
│           │ multiple tokens cause the neuron to activate strongly, the entire sequence will be contained between     │
│           │ bars | just like this|.                                                                                  │
│           │                                                                                                          │
│           │ You will be given multiple samples on which a neuron activates strongly. For each sample in turn, note   │
│           │ down a few features that the text possesses, even if you don't initially think they are important.       │
│           │                                                                                                          │
│           │ Once you have written down a few notes for each text sample, summarize what highly-activating samples    │
│           │ have in common. Finally, use your findings to produce a plausible explaination for what causes the       │
│           │ neuron to fire.                                                                                          │
│           │                                                                                                          │
│           │ Sample 0: broadcast.����While the episode will| mark| C.K.'s debut as an                                 │
│           │ Sample 1:����If implemented, this project would| mark| a turning point in the growing cooperation        │
│           │ between                                                                                                  │
│           │ Sample 2: States.����This year,which| marks| Fort Rossâ���� bicentennial,                                │
│           │                                                                                                          │
│ Assistant │ Based on the given text samples, I have noted down some features that the texts possess where the neuron │
│           │ strongly activates:                                                                                      │
│           │                                                                                                          │
│           │ 1. The presence of the word "marks" or its past tense form "marked." In all three samples, the neuron    │
│           │ activated strongly when it encountered this word.                                                        │
│           │ 2. The context of the word "marks" suggests a sense of significance or importance. In Sample 0, it       │
│           │ implies C.K.'s debut, in Sample 1, it signifies a turning point, and in Sample 2, it refers to a         │
│           │ bicentennial celebration.                                                                                │
│           │ 3. The presence of past tense verbs in general. All three samples contain past tense verbs, such as      │
│           │ "will mark," "if implemented," and "this year, which marks."                                             │
│      

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
